<a href="https://colab.research.google.com/github/marinaalfarias/ia940/blob/main/Waveshaping_e_S%C3%ADntese_Subtrativa.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt
import numpy as np
from IPython.display import Audio
import librosa
import librosa.display
import scipy.signal
fs = 44100

# Waveshaping

Implemente o circuito em questão usando taxa de amostragem de 44100 Hz e uma entrada de frequência 1 kHz.
Use as funções
`np.maximum()` e
`np.minimum()`.


In [ ]:
fs = 44100
T = 1
freq = 1000
t = np.arange(T*fs)/fs
x = np.cos(2 * np.pi * freq * t)

def waveshape(x):

  # Implemente o waveshaping aqui

  return y

y = waveshape(x)

# Acerte o eixo do tempo e coloque legendas nesta figura!
plt.figure(figsize=(10,10))
plt.plot(x, label='...')
plt.plot(y, label='...')
plt.legend()
plt.show()

##Defininindo função para análise espectral

Analise a Transformada de Fourier da saída. Ocorreu aliasing? Por que?


In [ ]:
def espectro(x,fs,fmin = 0,fmax= fs/2,ymin = -100, ymax=0):
  #Calcule a FFT do sinal janelado
  X = np.fft.fft(x * np.hanning(len(x)))
  #Compense o efeito da janela
  X /= np.sum(np.hanning(len(x)))/2
  #Extraia o módulo e calcule dB
  X = 20*np.log10(np.abs(X))

  #Grando veter de frequência
  F = np.arange(len(X)) * fs/ len(X)

  plt.figure()
  plt.plot(F,X)
  plt.xlim(fmin, fmax)
  plt.ylim(ymin,ymax)
  plt.ylabel('Magnitude [dB]')
  plt.xlabel('Frequência [Hz]')
  plt.show()

In [ ]:
# Mostre o espectro do sinal de entrada

# Mostre o espectro do sinal de saída

## Waveshape com anti-aliasing

In [ ]:
def waveshape_anti_aliasing(x, upsample=16):
  #sobreamostre a onda de entrada x
  x_ = np.ones(len(x)*upsample)
  for i in np.arange(len(x_)):
    x_[i] = x[int(i/upsample)]
  #aplique a distroção
  y_ = waveshape(x_)
  #decime o sinal de volta à taxa de amostragem original
  y = #decimate()
  return y

y = waveshape_anti_aliasing(x)

# Acerte o eixo do tempo e coloque legendas nesta figura!
plt.figure(figsize=(20,20))
plt.plot(x[0:400])
plt.plot(y[0:400])
plt.show()


In [ ]:
# Mostre o espectro do sinal de entrada

# Mostre o espectro do sinal de saída

# Formas de onda não-senoidais

In [ ]:
# Onda quadrada
t = np.arange(T*fs)/(fs)
x = scipy.signal.square(2 * np.pi * freq * t)

# Visualização no tempo
plt.figure()
plt.plot(x[0:400])
plt.show()

#Visualização na frequência
espectro(x,fs,0,fs,-50,0)

In [ ]:
# Onda quadrada
t = np.arange(T*fs)/(fs)
x = scipy.signal.sawtooth(2 * np.pi * freq * t, width = 0.5)

# Visualização no tempo
plt.figure()
plt.plot(x[0:400])
plt.show()

#Visualização na frequência
espectro(x,fs,0,fs,-100,0)

In [ ]:
def onda_quadrada(T, freq, fs, upsample=1):
  t = np.arange(T*fs*upsample)/(fs*upsample)
  x = scipy.signal.square(2 * np.pi * freq * t)

  if upsample > 1:
    x = scipy.signal.decimate(x, upsample)

  return x

plt.figure()
x = onda_quadrada(2, 440, fs, 10)
plt.plot(x[0:400])
plt.show()

In [ ]:
espectro(x,fs,0,fs,-50,0)

In [ ]:
def onda_triangular(T, freq, fs, upsample=1):
  t = np.arange(T*fs*upsample)/(fs*upsample)
  x = scipy.signal.sawtooth(2 * np.pi * freq * t,width = 0.5)

  if upsample > 1:
    x = scipy.signal.decimate(x, upsample)
  return x

plt.figure()
x = onda_triangular(2, 440, fs, upsample=10)
plt.plot(x[0:2000])
plt.show()

In [ ]:
espectro(x,fs,0,fs,-100,0)

In [ ]:
x = onda_quadrada(2, 440, fs, upsample=1)
x_ = onda_quadrada(2, 440, fs, upsample=10)
plt.plot(x[0:200])
plt.plot(x_[0:200])
plt.show()

plt.figure()
plt.plot(x[0:200]-x_[0:200])
plt.show()

In [ ]:
x = onda_triangular(2, 440, fs, upsample=1)
x_ = onda_triangular(2, 440, fs, upsample=10)
plt.plot(x[0:200])
plt.plot(x_[0:200])
plt.show()

plt.figure()
plt.plot(x[0:200]-x_[0:200])
plt.show()

# Filtro passa-banda com controle dinâmico de frequência central (VCF)

In [ ]:
# Ideia: aplicar filtro em pequenos quadros e preservar as condicoes iniciais.
# A cada novo quadro, re-estimar coreficientes do filtro.
def vcf(freqs, x, fs, frame_len, Q=5):
  y = np.zeros_like(x)
  i = 0
  zi = np.zeros(2)
  while i + frame_len < len(x):
    f = np.mean(freqs[i:i+frame_len])
    b, a = scipy.signal.iirpeak(w0=f, Q=Q, fs=fs)
    y[i:i+frame_len], zi = scipy.signal.lfilter(b, a, x[i:i+frame_len], zi=zi)
    i += frame_len
  return y

In [ ]:
x = np.random.random(fs*5)
t = np.arange(fs*5)/fs
freqs = 1500 + 1000 * np.cos(2 * np.pi * 440 * t)
y_ = vcf(freqs, x, fs, 8, 50)
freqs_ = 800 + 350 * np.cos(2 * np.pi * 5 * t)
y = vcf(freqs_, y_, fs, 8, 5)


In [ ]:
#Análise do sinal x

D = librosa.amplitude_to_db(np.abs(librosa.stft(x)),
                            ref=np.max)
librosa.display.specshow(D, y_axis='log', sr=fs, x_axis='time')

Audio(x, rate=fs)

In [ ]:
# Analise o sinal y_

In [ ]:
# Analise o sinal y